## STAC/ZARR 2025 workshop

This notebook shows the following features:
- CADIP STAC search
- AUXIP STAC search
- Staging of S1A CADIP session as a STAC item
- Staging of S1A auxiliary files (MPL_ORBPRE / MPL_ORBSCT) as STAC items

## 1. Initialization

In [ ]:
# Init environment before running a demo notebook.
import os
from resources.utils import *

init_demo()

# Reload the global vars again
from resources.utils import *

## 1. Find latest S1A CADIP session as a STAC item

In [ ]:
# Retrieve latest S1A session
cadip_session = cadip_client.search(method="GET", limit=1, max_items=1,
                                    collections=["s1_ins","s1_kse","s1_mps","s1_mti","s1_nsg","s1_sgs"],
                                    stac_filter="platform=sentinel-1a and cadip:delivery_push_ok=true",
                                    sortby=[ { "field": "datetime", "direction": "desc" } ])[0]
cadip_session_href = cadip_session.get_links("self")[0].get_href()
print(f"CADIP session: {cadip_session_href}")

## 2. Find auxiliary data from AUXIP as STAC items

In [ ]:
# Retrieve latest MPL_ORBPRE
# Here we show support of cql2-json filters in POST /search
mpl_orbpre = auxip_client.search(method="POST", limit=1, max_items=1, collections="S1-MPL_ORBPRE", stac_filter={
        "op": "and",
        "args": [
            { "op": "=", "args": [ { "property": "product:type" }, "MPL_ORBPRE" ] }
        ]
    }, sortby=[ { "field": "start_datetime", "direction": "desc" } ])[0]
mpl_orbpre_href = mpl_orbpre.get_links("self")[0].get_href()
print(f"MPL_ORBPRE: {mpl_orbpre_href}")

# Retrieve latest MPL_ORBSCT
mpl_orbsct = auxip_client.search(method="POST", limit=1, max_items=1, collections="S1-MPL_ORBSCT", stac_filter={
        "op": "and",
        "args": [
            { "op": "=", "args": [ { "property": "product:type" }, "MPL_ORBSCT" ] }
        ]
    }, sortby=[ { "field": "start_datetime", "direction": "desc" } ])[0]
mpl_orbsct_href = mpl_orbsct.get_links("self")[0].get_href()
print(f"MPL_ORBSCT: {mpl_orbsct_href}")

## 3. Create Catalog collections

In [ ]:
temporal = TemporalExtent([datetime(2014, 4, 3), datetime.now()])

s1_sessions_coll = get_or_create_test_collection(
    collection_id="s1_sessions", description="S1 staged CADIP sessions", title="S1 sessions", temporal=temporal)
mpl_orbpres_coll = get_or_create_test_collection(
    collection_id="s1_aux_mpl_orbpre", description="MPL_ORBPRE staged auxiliary files", title="MPL_ORBPRE", temporal=temporal)
mpl_orbscts_coll = get_or_create_test_collection(
    collection_id="s1_aux_mpl_orbsct", description="MPL_ORBSCT staged auxiliary files", title="MPL_ORBSCT", temporal=temporal)

print(f"S1 session collection: {s1_sessions_coll.get_links('self')[0].get_href()}")
print(f"MPL_ORBPRE collection: {mpl_orbpres_coll.get_links('self')[0].get_href()}")
print(f"MPL_ORBSCT collection: {mpl_orbscts_coll.get_links('self')[0].get_href()}")
print("STAC Browser: https://stac-browser-catalog.ops.rs-python.eu")

## 4. Stage session and auxiliary data in the STAC Catalog

In [ ]:
stage_single_item(mpl_orbpre, mpl_orbpres_coll)
stage_single_item(mpl_orbsct, mpl_orbscts_coll)
stage_single_item(cadip_session, s1_sessions_coll)